## Cell 1 — Install Dependencies

In [ ]:
!pip install -q transformers accelerate datasets tqdm
# 'accelerate' is needed for device_map='auto' GPU placement

## Cell 2 — Imports

In [1]:
import re
import json
import time
import random
import traceback
import torch
from tqdm.notebook import tqdm
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM

print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU             : {torch.cuda.get_device_name(0)}")
    print(f"VRAM            : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

PyTorch version : 2.10.0+cu128
CUDA available  : True
GPU             : Tesla T4
VRAM            : 15.6 GB


## Cell 3 — Configuration
✏️ **Edit the values in this cell before running.**

In [8]:
# ── Model ─────────────────────────────────────────────────
MODEL_ID    = "Qwen/Qwen2.5-3B-Instruct"

# ── Sampling ──────────────────────────────────────────────
TRIAL_N     = 1319
SEED        = 42

# ── Generation ────────────────────────────────────────────
MAX_NEW_TOKENS = 512
TEMPERATURE    = None

# ── Output ────────────────────────────────────────────────
OUTPUT_FILE = "gsm8k_qwen2_5_3b_results.json"

# ── System prompt ─────────────────────────────────────────
SYSTEM_PROMPT = (
    "You are a helpful math assistant. "
    "Solve the problem step by step, then end your answer with: "
    "'The answer is <number>.'"
)

print("Config set!")
print(f"  Model   : {MODEL_ID}")
print(f"  Trial N : {TRIAL_N or 'all 1319'}")
print(f"  Seed    : {SEED}")

Config set!
  Model   : Qwen/Qwen2.5-3B-Instruct
  Trial N : 1319
  Seed    : 42


## Cell 4 — Load Model & Tokenizer
Downloads Qwen2.5-3B-Instruct from HuggingFace and places it on the GPU.
First run will take ~2–3 minutes to download (~6 GB).

In [3]:
print(f"Loading tokenizer: {MODEL_ID} ...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

print(f"Loading model: {MODEL_ID} ...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,   # fp16 halves VRAM usage vs fp32
    device_map="auto",           # automatically places layers on GPU/CPU
    trust_remote_code=True,
)
model.eval()

DEVICE = next(model.parameters()).device
print(f"\nModel loaded on: {DEVICE}")
if torch.cuda.is_available():
    used_vram = torch.cuda.memory_allocated() / 1e9
    print(f"VRAM used       : {used_vram:.2f} GB")

Loading tokenizer: Qwen/Qwen2.5-3B-Instruct ...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]


Model loaded on: cuda:0
VRAM used       : 6.17 GB


## Cell 5 — Helper Functions

In [4]:
def extract_answer(text: str) -> str | None:
    """Extract the final numeric answer from model output or gold label."""
    # GSM8K gold format: #### <number>
    match = re.search(r"####\s*([\d,.\-]+)", text)
    if match:
        return match.group(1).replace(",", "").strip()

    # Model may say 'The answer is X'
    match = re.search(r"[Tt]he answer is[:\s]*([\d,.\-]+)", text)
    if match:
        return match.group(1).replace(",", "").strip()

    # Fallback: last number in text
    numbers = re.findall(r"[\d,]+\.?\d*", text)
    if numbers:
        return numbers[-1].replace(",", "").strip()

    return None


def is_correct(predicted: str | None, gold: str) -> bool:
    """Compare predicted and gold answers numerically."""
    if predicted is None:
        return False
    try:
        return abs(float(predicted) - float(gold)) < 1e-6
    except ValueError:
        return predicted.strip().lower() == gold.strip().lower()


def call_model(question: str) -> tuple[str, int, int, float]:
    """Tokenize a question, run local inference, and return
    (response_text, prompt_tokens, output_tokens, latency_seconds)."""
    # Build chat messages using the model's chat template
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": question},
    ]
    tokenized = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    )
    input_ids      = tokenized["input_ids"].to(DEVICE)
    attention_mask = tokenized["attention_mask"].to(DEVICE)
    prompt_tokens  = input_ids.shape[-1]

    # Build generation kwargs
    gen_kwargs = dict(
        input_ids=input_ids,
        attention_mask=attention_mask,
        max_new_tokens=MAX_NEW_TOKENS,
        pad_token_id=tokenizer.eos_token_id,
    )
    if TEMPERATURE is None:
        gen_kwargs["do_sample"] = False          # greedy
    else:
        gen_kwargs["do_sample"] = True
        gen_kwargs["temperature"] = TEMPERATURE

    t_start    = time.perf_counter()
    with torch.no_grad():
        output_ids = model.generate(**gen_kwargs)
    latency    = time.perf_counter() - t_start

    # Decode only the newly generated tokens (skip the prompt)
    new_tokens    = output_ids[0][input_ids.shape[-1]:]
    output_tokens = len(new_tokens)
    response_text = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    return response_text, prompt_tokens, output_tokens, latency


print("Helper functions defined.")

Helper functions defined.


## Cell 6 — Load & Sample Dataset

In [9]:
print("Loading GSM8K test split ...")
dataset   = load_dataset("openai/gsm8k", "main", split="test")
full_size = len(dataset)  # 1319

if TRIAL_N is not None:
    if TRIAL_N > full_size:
        raise ValueError(f"TRIAL_N ({TRIAL_N}) exceeds full test set size ({full_size}).")
    random.seed(SEED)
    sampled_indices = sorted(random.sample(range(full_size), TRIAL_N))
    dataset = dataset.select(sampled_indices)
    print(f"Randomly sampled {TRIAL_N} of {full_size} examples (seed={SEED}).")
else:
    sampled_indices = list(range(full_size))
    print(f"Using full test set ({full_size} examples).")

Loading GSM8K test split ...
Randomly sampled 1319 of 1319 examples (seed=42).


## Cell 7 — Run Benchmark

In [10]:
total         = len(dataset)
correct       = 0
errors        = 0
results       = []
total_latency        = 0.0
total_prompt_tokens  = 0
total_output_tokens  = 0

print(f"Running {total} examples ...\n")

for i, example in enumerate(tqdm(dataset, desc="Evaluating")):
    question = example["question"]
    gold_ans = extract_answer(example["answer"])

    try:
        response, prompt_tokens, output_tokens, latency = call_model(question)
        pred_ans     = extract_answer(response)
        correct_flag = is_correct(pred_ans, gold_ans)
    except Exception as exc:
        full_tb      = traceback.format_exc()
        response     = f"ERROR: {full_tb}"
        pred_ans     = None
        correct_flag = False
        prompt_tokens  = 0
        output_tokens  = 0
        latency        = 0.0
        errors        += 1
        print(f"\n[Error on example {i} (dataset_idx={sampled_indices[i]})]\n{full_tb}")

    if correct_flag:
        correct += 1

    total_latency       += latency
    total_prompt_tokens += prompt_tokens
    total_output_tokens += output_tokens
    tokens_per_sec       = output_tokens / latency if latency > 0 else 0.0

    results.append({
        "id":             i,
        "dataset_idx":    sampled_indices[i],
        "question":       question,
        "gold":           gold_ans,
        "predicted":      pred_ans,
        "correct":        correct_flag,
        "response":       response,
        "prompt_tokens":  prompt_tokens,
        "output_tokens":  output_tokens,
        "latency_s":      round(latency, 3),
        "tokens_per_sec": round(tokens_per_sec, 1),
    })

accuracy          = correct / total * 100
avg_latency       = total_latency / total
avg_output_tokens = total_output_tokens / total
avg_tokens_per_sec = total_output_tokens / total_latency if total_latency > 0 else 0.0

summary = {
    "model":             MODEL_ID,
    "trial_n":           TRIAL_N,
    "seed":              SEED,
    "total":             total,
    "correct":           correct,
    "errors":            errors,
    "accuracy":          round(accuracy, 2),
    "total_latency_s":   round(total_latency, 2),
    "avg_latency_s":     round(avg_latency, 3),
    "avg_output_tokens": round(avg_output_tokens, 1),
    "avg_tokens_per_sec":round(avg_tokens_per_sec, 1),
    "total_prompt_tokens": total_prompt_tokens,
    "total_output_tokens": total_output_tokens,
}

print("\n" + "=" * 45)
print(f"  Model              : {summary['model']}")
print(f"  Trial N            : {summary['trial_n'] or 'all'}")
print(f"  Seed               : {summary['seed']}")
print(f"  Total              : {summary['total']}")
print(f"  Correct            : {summary['correct']}")
print(f"  Errors             : {summary['errors']}")
print(f"  Accuracy           : {summary['accuracy']}%")
print(f"  Total latency      : {summary['total_latency_s']}s")
print(f"  Avg latency/sample : {summary['avg_latency_s']}s")
print(f"  Avg output tokens  : {summary['avg_output_tokens']}")
print(f"  Avg tokens/sec     : {summary['avg_tokens_per_sec']}")
print(f"  Total prompt tokens: {summary['total_prompt_tokens']}")
print(f"  Total output tokens: {summary['total_output_tokens']}")
print("=" * 45)

Running 1319 examples ...



Evaluating:   0%|          | 0/1319 [00:00<?, ?it/s]


  Model              : Qwen/Qwen2.5-3B-Instruct
  Trial N            : 1319
  Seed               : 42
  Total              : 1319
  Correct            : 1050
  Errors             : 0
  Accuracy           : 79.61%
  Total latency      : 20223.69s
  Avg latency/sample : 15.333s
  Avg output tokens  : 272.3
  Avg tokens/sec     : 17.8
  Total prompt tokens: 134944
  Total output tokens: 359213


## Cell 8 — Save & Download Results

In [7]:
output = {"summary": summary, "results": results}

with open(OUTPUT_FILE, "w") as f:
    json.dump(output, f, indent=2)

print(f"Results saved to: {OUTPUT_FILE}")

# Download the file directly in Colab
from google.colab import files
files.download(OUTPUT_FILE)

Results saved to: gsm8k_qwen2_5_3b_results.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Cell 9 — (Optional) Inspect Failures

In [ ]:
failures = [r for r in results if not r["correct"]]
print(f"Wrong answers: {len(failures)} / {total}\n")

# Print first 3 failures for a quick sanity check
for r in failures[:3]:
    print(f"--- Example {r['dataset_idx']} ---")
    print(f"Question  : {r['question'][:120]}...")
    print(f"Gold      : {r['gold']}")
    print(f"Predicted : {r['predicted']}")
    print(f"Response  : {r['response'][:200]}...")
    print()